In [ ]:
# Run this cell if you're missing any of these libraries
!pip install numpy xarray matplotlib cartopy seaborn netCDF4


In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import seaborn as sns
from netCDF4 import Dataset
import cartopy.feature as cfeature


In [ ]:
# NOAA GPCP Precipitation Dataset - Monthly Mean
# Data source: NOAA PSL (https://psl.noaa.gov/)
# No need to download manually — we read it directly from NOAA's THREDDS server

baseURL = 'http://www.esrl.noaa.gov'
catalogURL = '/psd/thredds/dodsC/Datasets/gpcp/precip.mon.mean.nc'
dataset_url = baseURL + catalogURL

# Open the dataset via NetCDF4 and convert to xarray
nc = Dataset(dataset_url)
precipID = xr.open_dataset(xr.backends.NetCDF4DataStore(nc))


In [ ]:
# Get the 'precip' variable from the dataset
precip = precipID['precip']

# Find the latest time step available in the dataset
mostRecent = len(precip.time.values) - 1

# Extract precipitation data for that time step
recentPrecip = precip.isel(time=mostRecent)


In [ ]:
# Set contour levels for the precipitation color scale
precipmin = 0
precipmax = 20
levels = np.linspace(precipmin, precipmax, 21)

# Choose seaborn's icefire color palette
cmap = sns.color_palette("icefire", as_cmap=True)

# Create the plot with an Orthographic projection (centered on Asia)
fig = plt.figure(figsize=[12, 6], facecolor='none')
ax = plt.subplot(1, 1, 1, projection=ccrs.Orthographic(central_longitude=90, central_latitude=0), facecolor='none')

# Plot the contour
contour = recentPrecip.plot.contourf(
    levels=levels,
    cmap=cmap,
    transform=ccrs.PlateCarree(),
    ax=ax,
    add_colorbar=False
)

# Add map features
ax.coastlines('10m')
ax.add_feature(cfeature.BORDERS, edgecolor='white')

# Optional: Add colorbar
cbar = plt.colorbar(contour, ax=ax, orientation='horizontal', pad=0.05)
cbar.set_label('Precipitation (mm/month)', fontsize=12)

# Optional: Add title
plt.title('Global Precipitation Climatology (Most Recent Month)', fontsize=14, weight='bold')

# Save the figure if needed
plt.savefig('precip_plot_asia_orthographic.png', dpi=300, bbox_inches='tight', transparent=True)

# Show the plot
plt.show()
